In [ ]:
import matplotlib as mpl
import seaborn as sns

sns.set_theme(style="ticks", context="paper", palette="deep")
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "figure.titlesize": 9,
    "figure.titleweight": "bold",
    "axes.titlesize": 8,
    "axes.titleweight": "normal",
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 6.5,
    "legend.title_fontsize": 7,
    "lines.linewidth": 1.4,
    "lines.markersize": 4,
    "patch.linewidth": 0.8,
    "axes.linewidth": 0.8,
    "xtick.major.size": 3.5,
    "ytick.major.size": 3.5,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.minor.size": 2,
    "ytick.minor.size": 2,
    "xtick.minor.width": 0.6,
    "ytick.minor.width": 0.6,
    "legend.frameon": False,
    "grid.linewidth": 0.5,
    "grid.alpha": 0.25,
    "figure.dpi": 150,
    "savefig.dpi": 600,
    "savefig.transparent": True,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

# Manuscript Figure 5 — Confinement and receptor density jointly reshape sensor response

This notebook analyzes the two 261016 sinusoidal-field sweeps. The full factorial dataset supplies complete background-aware occupancy and relative concentration-requirement curves. The fine amplitude sweep resolves $C_{10}$, $C_{50}$, $C_{90}$, and $C_{90}/C_{10}$ at 1 nm amplitude increments.

In [ ]:
from pathlib import Path
import importlib
import sys
import warnings

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "analysis_tools":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import analysis_tools.manuscript_escape_analysis as manuscript_escape_analysis
importlib.reload(manuscript_escape_analysis)
from analysis_tools.manuscript_escape_analysis import (
    discover_runs, save_figure, size_figure_for_axes,
)

SAVE_FIGURES = True
REFRESH_CACHE = False
SHOW_PANEL_TITLES = False
FIGURE_ROOT = PROJECT_ROOT / "analysis_tools" / "manuscript_panels" / "figure 5"
CACHE_ROOT = PROJECT_ROOT / "analysis_tools" / ".analysis_cache"
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

ONEDRIVE_RESULTS_ROOT = Path(
    "/Users/rynon/Library/CloudStorage/OneDrive-NorthwesternUniversity/PhD/Data/"
    "Agent-based simulations"
)
RESULTS_ROOT = (
    ONEDRIVE_RESULTS_ROOT if ONEDRIVE_RESULTS_ROOT.exists()
    else PROJECT_ROOT / "results"
)
print(f"Simulation results root: {RESULTS_ROOT}")

# Physical size of the axes data rectangle. Canvas margins grow around it.
PLOT_AREA_SIZE_IN = (2.55, 2.15)
THREE_PANEL_PLOT_AREA_IN = (1.75, 2.15)
SINGLE_PANEL_SIZE_IN = (3.4, 2.8)

def finish_panel(fig, ax, plot_area=PLOT_AREA_SIZE_IN, padding_in=0.08):
    if not SHOW_PANEL_TITLES:
        for current_ax in fig.axes:
            current_ax.set_title("")
    size_figure_for_axes(fig, ax, *plot_area, padding_in=padding_in)

def save_panel(fig, stem, dpi=None):
    save_figure(
        fig, FIGURE_ROOT, stem, dpi=dpi, exact_size=True, transparent=True,
    )

def add_panel_letter(ax, letter):
    ax.text(
        -0.20, 1.06, letter, transform=ax.transAxes, fontsize=9,
        fontweight="bold", va="top",
    )

## 1. Discover and summarize the 261016 sweeps

The first uncached run reads only the compact receptor-summary tables. Subsequent runs use local CSV caches. Set `REFRESH_CACHE = True` only after changing or replacing the underlying simulation results.

In [ ]:
DATASETS = {
    "factorial": (
        "261016_manuscript_fig5_sinusoidal_factorial_response", 3672,
    ),
    "amplitude": (
        "261016_manuscript_fig5_sinusoidal_amplitude_width_sweep", 2232,
    ),
}

def summarize_from_receptor_tables(index, cache_path, refresh=False):
    cache_path = Path(cache_path)
    if cache_path.exists() and not refresh:
        cached = pd.read_csv(cache_path)
        if len(cached) == len(index):
            return cached
        warnings.warn(f"Rebuilding incomplete cache: {cache_path}")

    rows = []
    columns = [
        "n_trials", "n_escaped", "mean_bound_time_s",
        "mean_rebindings", "probability_any_rebinding",
    ]
    for number, run in enumerate(index.to_dict("records"), start=1):
        receptor = pd.read_csv(
            Path(run["run_directory"]) / "receptor_summary.csv",
            usecols=columns,
        )
        weights = receptor.n_trials.to_numpy(float)
        total_trials = float(weights.sum())
        def weighted(column):
            return float(np.average(receptor[column].to_numpy(float), weights=weights))
        summary = dict(run)
        summary.update({
            "n_locations": len(receptor),
            "n_trials": int(total_trials),
            "escape_probability": float(receptor.n_escaped.sum()/total_trials),
            "censoring_fraction": float(1-receptor.n_escaped.sum()/total_trials),
            "mean_bound_time_s": weighted("mean_bound_time_s"),
            "mean_rebindings": weighted("mean_rebindings"),
            "probability_any_rebinding": weighted("probability_any_rebinding"),
        })
        rows.append(summary)
        if number % 250 == 0 or number == len(index):
            print(f"  summarized {number:,}/{len(index):,} runs", end="\r")
    print()
    result = pd.DataFrame(rows)
    result.to_csv(cache_path, index=False)
    return result

indexes = {}
summaries = {}
for key, (folder, expected) in DATASETS.items():
    indexes[key] = discover_runs(
        RESULTS_ROOT / folder,
        cache_path=CACHE_ROOT / f"figure5_261016_{key}_index.csv",
        refresh_cache=REFRESH_CACHE,
    )
    summaries[key] = summarize_from_receptor_tables(
        indexes[key],
        CACHE_ROOT / f"figure5_261016_{key}_replicates.csv",
        refresh=REFRESH_CACHE,
    )
    print(f"{key}: {len(indexes[key]):,}/{expected:,} completed runs")
    if len(indexes[key]) != expected:
        warnings.warn(f"The {key} sweep is incomplete.")

## 2. Convert residence time to background-aware concentration requirement

For imposed occupancy $\theta$, the normalized concentration requirement and its classical ratio are

$$\frac{C(\theta)}{K_D}=\frac{\theta}{(1-\theta)k_{off}\tau_{B,eff}(\theta)},\qquad R_C(\theta)=\frac{1}{k_{off}\tau_{B,eff}(\theta)}.$$

In [ ]:
def prepare_response(frame):
    result = frame.copy()
    theta = result.background_occupancy_fraction.to_numpy(float)
    denominator = (
        (1-theta) * result.k_off_s.to_numpy(float)
        * result.mean_bound_time_s.to_numpy(float)
    )
    result["theta"] = theta
    result["C_over_KD"] = np.divide(
        theta, denominator, out=np.full(len(result), np.nan),
        where=denominator > 0,
    )
    result["classical_C_over_KD"] = theta/(1-theta)
    result["R_C"] = result.C_over_KD/result.classical_C_over_KD
    result["condition_key"] = list(zip(
        result.amplitude_nm.astype(float),
        result.receptor_density_m2.astype(float),
    ))
    return result

factorial = prepare_response(summaries["factorial"])
amplitude = prepare_response(summaries["amplitude"])

def response_statistics(frame):
    return (
        frame.groupby(["amplitude_nm", "receptor_density_m2", "theta"])
        .agg(
            C_mean=("C_over_KD", "mean"),
            C_sem=("C_over_KD", "sem"),
            R_C_mean=("R_C", "mean"),
            R_C_sem=("R_C", "sem"),
        ).reset_index()
    )

factorial_stats = response_statistics(factorial)

def calculate_three_point_widths(frame):
    pivot = frame.pivot_table(
        index=["amplitude_nm", "receptor_density_m2", "replicate"],
        columns="theta", values="C_over_KD", aggfunc="first",
    ).reset_index()
    required = [0.1, 0.5, 0.9]
    missing = [value for value in required if value not in pivot.columns]
    if missing:
        raise ValueError(f"Missing occupancy states in fine sweep: {missing}")
    return pivot.rename(columns={
        0.1: "C10_over_KD", 0.5: "C50_over_KD", 0.9: "C90_over_KD",
    }).assign(C90_over_C10=lambda value: value.C90_over_KD/value.C10_over_KD)

amplitude_widths = calculate_three_point_widths(amplitude)

qc = pd.DataFrame([
    {"dataset": "factorial", "runs": len(factorial),
     "amplitudes": factorial.amplitude_nm.nunique(),
     "densities": factorial.receptor_density_m2.nunique(),
     "occupancy_states": factorial.theta.nunique(),
     "maximum_censoring": factorial.censoring_fraction.max()},
    {"dataset": "fine amplitude", "runs": len(amplitude),
     "amplitudes": amplitude.amplitude_nm.nunique(),
     "densities": amplitude.receptor_density_m2.nunique(),
     "occupancy_states": amplitude.theta.nunique(),
     "maximum_censoring": amplitude.censoring_fraction.max()},
])
display(qc)

## 3. Load self- and cross-rebinding at the response endpoints

Only the $\theta=0.001$ and $0.999$ trajectory archives are read. Chunked aggregation limits memory use for the highest-density surfaces.

In [ ]:
EXTREME_OCCUPANCIES = (0.001, 0.999)

def summarize_extreme_rebindings(index, cache_path, refresh=False):
    occupancy = index.background_occupancy_fraction.to_numpy(float)
    selected = index.loc[
        np.isclose(occupancy, EXTREME_OCCUPANCIES[0])
        | np.isclose(occupancy, EXTREME_OCCUPANCIES[1])
    ]
    cache_path = Path(cache_path)
    required_columns = {
        "mean_self_rebindings", "mean_cross_rebindings",
        "background_occupancy_fraction",
    }
    if cache_path.exists() and not refresh:
        cached = pd.read_csv(cache_path)
        if len(cached) == len(selected) and required_columns.issubset(cached.columns):
            return cached
        warnings.warn(f"Rebuilding incomplete endpoint cache: {cache_path}")

    rows = []
    for number, run in enumerate(selected.to_dict("records"), start=1):
        count = 0
        self_total = 0.0
        cross_total = 0.0
        trajectory_path = Path(run["run_directory"]) / "trajectories.csv.gz"
        for chunk in pd.read_csv(
            trajectory_path,
            usecols=["n_self_rebindings", "n_cross_rebindings"],
            chunksize=250_000,
        ):
            count += len(chunk)
            self_total += float(chunk.n_self_rebindings.sum())
            cross_total += float(chunk.n_cross_rebindings.sum())
        row = dict(run)
        row.update({
            "n_trajectories_loaded": count,
            "mean_self_rebindings": self_total/count,
            "mean_cross_rebindings": cross_total/count,
            "self_rebinding_events": self_total,
            "cross_rebinding_events": cross_total,
        })
        rows.append(row)
        if number % 12 == 0 or number == len(selected):
            print(f"  endpoint rebindings: {number}/{len(selected)}", end="\r")
    print()
    result = pd.DataFrame(rows)
    result.to_csv(cache_path, index=False)
    return result

extreme_rebinding = summarize_extreme_rebindings(
    indexes["factorial"],
    CACHE_ROOT / "figure5_261016_factorial_extreme_rebindings.csv",
    refresh=REFRESH_CACHE,
)
# Each trajectory contributes one initial free excursion; every recorded
# rebinding creates another eventual free excursion. These probabilities
# therefore normalize cumulative counts by the number of opportunities.
excursion_denominator = (
    extreme_rebinding.n_trajectories_loaded
    + extreme_rebinding.self_rebinding_events
    + extreme_rebinding.cross_rebinding_events
)
extreme_rebinding["self_probability_per_excursion"] = (
    extreme_rebinding.self_rebinding_events/excursion_denominator
)
extreme_rebinding["cross_probability_per_excursion"] = (
    extreme_rebinding.cross_rebinding_events/excursion_denominator
)
print(f"Loaded {len(extreme_rebinding)} endpoint replicate summaries.")

## 4. Plotting definitions

In [ ]:
DENSITIES = [1e15, 1e16, 1e17]
DENSITY_NAMES = {
    1e15: r"$10^{15}$ m$^{-2}$",
    1e16: r"$10^{16}$ m$^{-2}$",
    1e17: r"$10^{17}$ m$^{-2}$",
}
DENSITY_COLORS = dict(zip(
    DENSITIES, plt.cm.viridis(np.linspace(.16, .88, len(DENSITIES)))
))
CORNERS = [(0.0, 1e15), (0.0, 1e17), (30.0, 1e15), (30.0, 1e17)]
CORNER_NAMES = {
    (0.0, 1e15): "Planar, low density",
    (0.0, 1e17): "Planar, high density",
    (30.0, 1e15): "High amplitude, low density",
    (30.0, 1e17): "High amplitude, high density",
}
CORNER_COLORS = {
    (0.0, 1e15): "#8C8C8C",
    (0.0, 1e17): "#377EB8",
    (30.0, 1e15): "#E68613",
    (30.0, 1e17): "#7B3294",
}
LOGIT_TICKS = [0.001, 0.01, 0.1, 0.5, 0.9, 0.99, 0.999]

def corner_subset(frame, condition):
    amplitude_nm, density = condition
    return frame.loc[
        np.isclose(frame.amplitude_nm, amplitude_nm)
        & np.isclose(frame.receptor_density_m2, density)
    ].sort_values("theta")

def plot_occupancy_panel(stats, letter, stem):
    fig, ax = plt.subplots(figsize=SINGLE_PANEL_SIZE_IN, constrained_layout=True)
    for condition in CORNERS:
        group = corner_subset(stats, condition)
        color = CORNER_COLORS[condition]
        ax.plot(group.C_mean, group.theta, color=color, label=CORNER_NAMES[condition])
        ax.fill_betweenx(
            group.theta, np.maximum(group.C_mean-group.C_sem, np.finfo(float).tiny),
            group.C_mean+group.C_sem, color=color, alpha=.16, linewidth=0,
        )
    theta_grid = np.linspace(.001, .999, 1000)
    ax.plot(theta_grid/(1-theta_grid), theta_grid, "k--", label="Classical")
    ax.set(
        xscale="log", xlim=(1e-5, 2e3), ylim=(0, 1),
        xlabel=r"Normalized concentration $C/K_D$", ylabel=r"Occupancy $\theta$",
        title="Factorial response extremes",
    )
    ax.legend(loc="upper left", fontsize=5.6, handlelength=1.5, labelspacing=.3)

    dilute_ax = ax.inset_axes([.08, .10, .31, .26])
    saturated_ax = ax.inset_axes([.65, .66, .31, .26])
    for condition in CORNERS:
        group = corner_subset(stats, condition)
        dilute = group.loc[group.theta.le(.1)]
        saturated = group.loc[group.theta.ge(.9)]
        dilute_ax.plot(dilute.C_mean, dilute.theta, color=CORNER_COLORS[condition])
        saturated_ax.plot(
            saturated.C_mean, saturated.theta, color=CORNER_COLORS[condition]
        )
    dilute_theta = np.geomspace(.001, .1, 300)
    saturated_theta = 1-np.geomspace(.001, .1, 300)[::-1]
    dilute_ax.plot(dilute_theta/(1-dilute_theta), dilute_theta, "k--")
    saturated_ax.plot(
        saturated_theta/(1-saturated_theta), saturated_theta, "k--"
    )
    dilute_ax.set(xscale="log", xlim=(1e-5, .2), ylim=(0, .105), ylabel=r"$\theta$")
    saturated_ax.set(xscale="log", xlim=(3, 1.5e3), ylim=(.9, 1), ylabel=r"$\theta$")
    saturated_ax.set_yticks([.90, .95, 1.00], ["0.90", "0.95", "1.00"])
    dilute_ax.text(.5, 1.04, "Dilute", transform=dilute_ax.transAxes, ha="center", fontsize=5.5)
    saturated_ax.text(.5, 1.04, "Near saturation", transform=saturated_ax.transAxes, ha="center", fontsize=5.5)
    for inset in (dilute_ax, saturated_ax):
        inset.tick_params(labelsize=4.5, pad=1.2)
        inset.yaxis.label.set_size(5.5)
    add_panel_letter(ax, letter)
    finish_panel(fig, ax)
    if SAVE_FIGURES: save_panel(fig, stem)
    plt.show()

def plot_requirement_panel(stats, letter, stem):
    fig, ax = plt.subplots(figsize=SINGLE_PANEL_SIZE_IN, constrained_layout=True)
    for condition in CORNERS:
        group = corner_subset(stats, condition)
        color = CORNER_COLORS[condition]
        ax.plot(group.theta, group.R_C_mean, color=color, label=CORNER_NAMES[condition])
        ax.fill_between(
            group.theta, np.maximum(group.R_C_mean-group.R_C_sem, np.finfo(float).tiny),
            group.R_C_mean+group.R_C_sem, color=color, alpha=.16, linewidth=0,
        )
    ax.axhline(1, color="black", linestyle="--", label="Classical")
    ax.set_xscale("logit")
    ax.set_xlim(.001, .999)
    ax.set_xticks(LOGIT_TICKS, [f"{tick:g}" for tick in LOGIT_TICKS])
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
    ax.set(
        xlabel=r"Occupancy $\theta$", ylabel=r"Relative concentration requirement $R_C$",
        title="Concentration requirement across occupancy",
    )
    # Reserve space above the classical limit for the legend.
    ax.set_ylim(0, 1.32)
    ax.legend(
        loc="upper left", ncol=1, fontsize=5.5,
        handlelength=1.4, labelspacing=.25, columnspacing=.8,
    )
    add_panel_letter(ax, letter)
    finish_panel(fig, ax, plot_area=THREE_PANEL_PLOT_AREA_IN)
    if SAVE_FIGURES: save_panel(fig, stem)
    plt.show()

def plot_c50_heatmap(frame, letter, stem):
    c50 = frame.loc[np.isclose(frame.theta, .5)]
    summary = (
        c50.groupby(["amplitude_nm", "receptor_density_m2"]).C_over_KD
        .agg(["mean", "sem"]).reset_index()
    )
    matrix = summary.pivot(index="receptor_density_m2", columns="amplitude_nm", values="mean")
    annotations = matrix.map(lambda value: f"{value:.3g}")
    fig, ax = plt.subplots(figsize=SINGLE_PANEL_SIZE_IN, constrained_layout=True)
    sns.heatmap(
        matrix, annot=annotations, fmt="", cmap="magma_r",
        norm=LogNorm(vmin=matrix.to_numpy().min(), vmax=1.0),
        cbar_kws={"label": r"$C_{50}/K_D$"}, ax=ax,
    )
    ax.set(
        xlabel="Sinusoidal amplitude (nm)", ylabel=r"Receptor density (m$^{-2}$)",
        title=r"Factorial $C_{50}/K_D$",
    )
    ax.set_yticklabels([r"$10^{15}$", r"$10^{16}$", r"$10^{17}$"], rotation=0)
    add_panel_letter(ax, letter)
    finish_panel(fig, ax)
    if SAVE_FIGURES: save_panel(fig, stem)
    plt.show()
    return summary

def plot_amplitude_width(frame, letter, stem):
    stats = (
        frame.groupby(["amplitude_nm", "receptor_density_m2"]).C90_over_C10
        .agg(["mean", "sem"]).reset_index()
    )
    fig, ax = plt.subplots(figsize=SINGLE_PANEL_SIZE_IN, constrained_layout=True)
    for density in DENSITIES:
        group = stats.loc[np.isclose(stats.receptor_density_m2, density)].sort_values("amplitude_nm")
        color = DENSITY_COLORS[density]
        ax.plot(group.amplitude_nm, group["mean"], color=color, label=DENSITY_NAMES[density])
        ax.fill_between(
            group.amplitude_nm, np.maximum(group["mean"]-group["sem"], np.finfo(float).tiny),
            group["mean"]+group["sem"], color=color, alpha=.16, linewidth=0,
        )
    ax.axhline(81, color="black", linestyle="--", label="Classical")
    ax.set(
        xlim=(0, 30), xlabel="Sinusoidal amplitude (nm)",
        ylabel=r"Response width $C_{90}/C_{10}$",
        title="Amplitude-dependent response width",
    )
    ax.legend(fontsize=5.8)
    add_panel_letter(ax, letter)
    finish_panel(fig, ax, plot_area=THREE_PANEL_PLOT_AREA_IN)
    if SAVE_FIGURES: save_panel(fig, stem)
    plt.show()
    return stats

def plot_endpoint_rebindings(frame, component, letter, stem):
    metric = f"mean_{component}_rebindings"
    stats = (
        frame.groupby(["amplitude_nm", "receptor_density_m2", "background_occupancy_fraction"])[metric]
        .agg(["mean", "sem"]).reset_index()
    )
    positions = np.arange(len(CORNERS))
    width = .34
    styles = {
        .001: {"color": "#2878B5", "hatch": "", "offset": -width/2},
        .999: {"color": "#D55E00", "hatch": "///", "offset": width/2},
    }
    fig, ax = plt.subplots(figsize=SINGLE_PANEL_SIZE_IN, constrained_layout=True)
    for occupancy in EXTREME_OCCUPANCIES:
        values = []
        errors = []
        for amplitude_nm, density in CORNERS:
            row = stats.loc[
                np.isclose(stats.amplitude_nm, amplitude_nm)
                & np.isclose(stats.receptor_density_m2, density)
                & np.isclose(stats.background_occupancy_fraction, occupancy)
            ]
            values.append(float(row["mean"].iloc[0]))
            errors.append(float(row["sem"].iloc[0]))
        style = styles[occupancy]
        ax.bar(
            positions+style["offset"], values, width, yerr=errors,
            color=style["color"], hatch=style["hatch"], edgecolor="0.2",
            linewidth=.7, capsize=2, label=rf"$\theta={occupancy:g}$",
        )
    ax.set_xticks(positions, [CORNER_NAMES[key] for key in CORNERS])
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
    if component == "cross":
        ax.set_yscale("symlog", linthresh=1e-4, linscale=.8)
        ylabel = "Mean cross-rebindings per trajectory"
    else:
        ax.set_yscale("log")
        ylabel = "Mean self-rebindings per trajectory"
    ax.set(ylabel=ylabel, title=f"{component.title()}-rebinding at response extremes")
    ax.legend(ncol=2, loc="lower center", bbox_to_anchor=(.5, 1.02))
    add_panel_letter(ax, letter)
    finish_panel(fig, ax, plot_area=THREE_PANEL_PLOT_AREA_IN)
    if SAVE_FIGURES: save_panel(fig, stem)
    plt.show()
    return stats

def plot_rebinding_outcome_proportions(frame, letter, stem):
    metrics = ["self_probability_per_excursion", "cross_probability_per_excursion"]
    stats = (
        frame.groupby(["amplitude_nm", "receptor_density_m2", "background_occupancy_fraction"])[metrics]
        .agg(["mean", "sem"]).reset_index()
    )
    stats.columns = [
        "amplitude_nm", "receptor_density_m2",
        "background_occupancy_fraction", "self_mean", "self_sem",
        "cross_mean", "cross_sem",
    ]
    positions = np.arange(len(CORNERS))
    occupancy_colors = {.001: "#2878B5", .999: "#D55E00"}
    component_styles = {
        "self": {"marker": "o", "linestyle": "-", "offset": -.04},
        "cross": {"marker": "s", "linestyle": "--", "offset": .04},
    }
    fig, ax = plt.subplots(figsize=SINGLE_PANEL_SIZE_IN, constrained_layout=True)
    for occupancy in EXTREME_OCCUPANCIES:
        self_values, self_errors = [], []
        cross_values, cross_errors = [], []
        for amplitude_nm, density in CORNERS:
            row = stats.loc[
                np.isclose(stats.amplitude_nm, amplitude_nm)
                & np.isclose(stats.receptor_density_m2, density)
                & np.isclose(stats.background_occupancy_fraction, occupancy)
            ]
            self_values.append(float(row.self_mean.iloc[0]))
            self_errors.append(float(row.self_sem.iloc[0]))
            cross_values.append(float(row.cross_mean.iloc[0]))
            cross_errors.append(float(row.cross_sem.iloc[0]))
        for component, values, errors in (
            ("self", self_values, self_errors),
            ("cross", cross_values, cross_errors),
        ):
            style = component_styles[component].copy()
            x = positions+style.pop("offset")
            ax.errorbar(
                x, values, yerr=errors, color=occupancy_colors[occupancy],
                linewidth=1.1, markersize=3.5, capsize=2, **style,
            )
    ax.set_xticks(positions, [CORNER_NAMES[key] for key in CORNERS])
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
    ax.set_yscale("log")
    ax.set_ylim(5e-5, 1)
    ax.set_ylabel("Rebinding probability per free excursion")
    ax.set_title("Rebinding outcome probabilities")
    ax.legend(
        handles=[
            Line2D([0], [0], color="#2878B5", label=r"$\theta=0.001$"),
            Line2D([0], [0], color="#D55E00", label=r"$\theta=0.999$"),
            Line2D([0], [0], color="0.2", marker="o", linestyle="-", label="Self"),
            Line2D([0], [0], color="0.2", marker="s", linestyle="--", label="Cross"),
        ],
        ncol=2, loc="lower center", bbox_to_anchor=(.5, 1.02),
        columnspacing=.8, handletextpad=.4,
    )
    add_panel_letter(ax, letter)
    finish_panel(fig, ax, plot_area=THREE_PANEL_PLOT_AREA_IN)
    if SAVE_FIGURES: save_panel(fig, stem)
    plt.show()
    return stats

## Figure 5A — Full occupancy curves for the factorial corners

In [ ]:
plot_occupancy_panel(
    factorial_stats, "A", "figure5_A_factorial_occupancy_curves",
)

## Figure 5B — Factorial $C_{50}/K_D$ landscape

In [ ]:
factorial_c50_summary = plot_c50_heatmap(
    factorial, "B", "figure5_B_factorial_C50_heatmap",
)

## Figure 5C — Self- and cross-rebinding probability per free excursion

In [ ]:
rebinding_probability_summary = plot_rebinding_outcome_proportions(
    extreme_rebinding, "C", "figure5_C_rebinding_outcome_probabilities",
)

## Figure 5D — Relative concentration requirement across occupancy

In [ ]:
plot_requirement_panel(
    factorial_stats, "D", "figure5_D_factorial_relative_concentration",
)

## Figure 5E — Fine amplitude sweep of $C_{90}/C_{10}$

In [ ]:
amplitude_width_summary = plot_amplitude_width(
    amplitude_widths, "E", "figure5_E_amplitude_C90_over_C10",
)

## Diagnostics and exported source tables

These tables expose the replicate-level values used in every panel and make censoring or missing-condition problems visible before figure assembly.

In [ ]:
display(factorial_c50_summary)
display(amplitude_width_summary.head(12))
display(
    extreme_rebinding.groupby(
        ["amplitude_nm", "receptor_density_m2", "background_occupancy_fraction"]
    )[["mean_self_rebindings", "mean_cross_rebindings"]].agg(["mean", "sem"])
)

if SAVE_FIGURES:
    output = FIGURE_ROOT / "figure5_tables"
    output.mkdir(parents=True, exist_ok=True)
    factorial.to_csv(output / "figure5_factorial_response_replicates.csv", index=False)
    factorial_stats.to_csv(output / "figure5_factorial_response_summary.csv", index=False)
    factorial_c50_summary.to_csv(output / "figure5_factorial_C50_summary.csv", index=False)
    amplitude.to_csv(output / "figure5_amplitude_response_replicates.csv", index=False)
    amplitude_widths.to_csv(output / "figure5_amplitude_width_replicates.csv", index=False)
    amplitude_width_summary.to_csv(output / "figure5_amplitude_width_summary.csv", index=False)
    extreme_rebinding.to_csv(output / "figure5_endpoint_rebinding_replicates.csv", index=False)
    rebinding_probability_summary.to_csv(
        output / "figure5_rebinding_outcome_probability_summary.csv", index=False
    )
    print(f"Saved panels and source tables to {FIGURE_ROOT}")